In [0]:
%run /Workspace/Users/antoniorad15@gmail.com/ROBOTICS-AI-training-pipeline/pipeline-finetune-gr00t/secrets-template

In [0]:
TMUX_COUNTER = 0

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_4", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_4", 0o600)

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
echo == OS ==
. /etc/os-release
echo $PRETTY_NAME
echo == GPU / Driver ==
nvidia-smi
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
curl -LsSf https://astral.sh/uv/install.sh | /bin/sh;
uv --version;
sudo apt-get update
sudo apt-get install -y libglu1-mesa tmux
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
git clone https://github.com/REBELDOT-SOLUTIONS-S-R-L/ROBOTICS-lehome-challenge.git
EOF

In [0]:
%sh
# Ctrl+C the current command, then:
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
# kill any hanging hf processes
pkill -f "hf download" || true
pkill -f "huggingface" || true

# remove stale lock files
find ~/ROBOTICS-lehome-challenge/Assets/.cache/huggingface/download -name "*.lock" -delete

# re-run the download
cd ROBOTICS-lehome-challenge
source .venv/bin/activate
hf download lehome/asset_challenge --repo-type dataset --local-dir Assets
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
cd ROBOTICS-lehome-challenge
uv sync
git clone -b lehome-cloth-mimic-compat https://github.com/alex-luci/IsaacLab.git third_party/IsaacLab
#test
#git checkout fix/cuda-env-support

git checkout auto-annotation-teleop
git fetch
git pull
git status
#git reset --hard ab8230758ed5ffd2901b9ebc38f0e097e944b537
source .venv/bin/activate
Yes | ./third_party/IsaacLab/isaaclab.sh -i none
uv pip install -e ./source/lehome
hf download lehome/asset_challenge --repo-type dataset --local-dir Assets
EOF
scp -r -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/assets/so101_follower_eef.usd \
  ubuntu@$BREV_IP:~/ROBOTICS-lehome-challenge/Assets/robots/lerobot/

In [0]:
# === Pool configuration ===
MAX_CONCURRENT = 3            # max parallel tmux sessions the instance can handle
INPUT_DIR = "~/mimicgen_dataset"  # remote folder containing annotated HDF5 files
SESSION_PREFIX = "generate"
SSH_KEY = "/tmp/ssh_private_key_4"
POLL_INTERVAL = 30            # seconds between status checks

In [0]:
import subprocess, time, os

BREV_IP = os.environ["BREV_IP"]

def ssh(cmd):
    r = subprocess.run(
        ["ssh", "-i", SSH_KEY, "-o", "StrictHostKeyChecking=no", f"ubuntu@{BREV_IP}", cmd],
        capture_output=True, text=True
    )
    return r.stdout.strip(), r.returncode

# --- discover input files ---
out, _ = ssh(f"ls {INPUT_DIR}/*.hdf5 2>/dev/null | sort")
files = [f for f in out.split("\n") if f.strip()]
print(f"Found {len(files)} input files:")
for f in files:
    print(f"  {f}")

def active_sessions():
    out, rc = ssh("tmux list-sessions -F '#{session_name}' 2>/dev/null || true")
    if rc != 0 or not out:
        return set()
    return {s for s in out.split("\n") if s.startswith(SESSION_PREFIX + "_")}

GARMENT_NAMES = ["Top_Long_Seen_0", "Top_Long_Seen_1", "Top_Long_Seen_2"]

def launch(filepath, garment_name):
    stem = filepath.split("/")[-1].replace(".hdf5", "")
    sname = f"{SESSION_PREFIX}_{garment_name}"
    out_stem = f"generated_dataset_{garment_name}"

    cmd = f"""
tmux kill-session -t {sname} 2>/dev/null || true
sleep 2
tmux new-session -d -s {sname}
tmux send-keys -t {sname} 'cd ROBOTICS-lehome-challenge && source .venv/bin/activate && yes Yes | python scripts/mimicgen/generate_dataset.py \
  --task LeHome-BiSO101-ManagerBased-Garment-Mimic-v0 \
  --garment_name {garment_name} \
  --input_file {filepath} \
  --output_file Datasets/hdf5_datasets/4_generated_datasets/{out_stem}.hdf5 \
  --generation_num_trials 200 \
  --device cuda \
  --num_envs 1 \
  --enable_cameras \
  --logging_interval 10 \
  --log_success \
  --headless ; sleep 5 ; tmux kill-session -t {sname}' Enter
"""
    ssh(cmd)
    print(f"  [STARTED] {sname}  ({filepath}, {garment_name})")
    return sname

# --- build queue as (file, garment) pairs ---
queue = [(f, g) for f in files for g in GARMENT_NAMES]

launched = set()
completed = set()

print(f"\nStarting pool (MAX_CONCURRENT={MAX_CONCURRENT}, {len(queue)} jobs)...")

while queue and len(launched - completed) < MAX_CONCURRENT:
    f, g = queue.pop(0)
    launched.add(launch(f, g))

while launched - completed:
    time.sleep(POLL_INTERVAL)
    active = active_sessions()
    for s in list(launched - completed - active):
        print(f"  [DONE] {s}")
        completed.add(s)
    while queue and len(launched - completed) < MAX_CONCURRENT:
        f, g = queue.pop(0)
        launched.add(launch(f, g))
    print(f"  active={len(launched - completed)}  done={len(completed)}  queued={len(queue)}")

print(f"\nAll {len(completed)} sessions completed.")

In [0]:
import subprocess, time, os

BREV_IP = os.environ["BREV_IP"]
SSH_KEY = "/tmp/ssh_private_key_4"
INPUT_CONVERT_DIR = "~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/4_generated_datasets"
OUTPUT_DIR = "~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/5_final_generated_datasets"
CONVERT_PREFIX = "convert"
MAX_CONCURRENT_CONVERT = 6
POLL_INTERVAL_CONVERT = 15

def ssh(cmd):
    r = subprocess.run(
        ["ssh", "-i", SSH_KEY, "-o", "StrictHostKeyChecking=no", f"ubuntu@{BREV_IP}", cmd],
        capture_output=True, text=True
    )
    return r.stdout.strip(), r.returncode

# discover good files
out, _ = ssh(f"ls {INPUT_CONVERT_DIR}/*.hdf5 2>/dev/null | grep -v '_failed' | sort")
files = [f for f in out.split("\n") if f.strip()]
print(f"Found {len(files)} files to convert:")
for f in files:
    print(f"  {f}")

def active_sessions():
    out, rc = ssh("tmux list-sessions -F '#{session_name}' 2>/dev/null || true")
    if rc != 0 or not out:
        return set()
    return {s for s in out.split("\n") if s.startswith(CONVERT_PREFIX + "_")}

def launch_convert(filepath):
    stem = filepath.split("/")[-1].replace(".hdf5", "")
    out_stem = stem.replace("generated_dataset_home_pos_test_", "converted_")
    sname = f"{CONVERT_PREFIX}_{stem}"
    cmd = f"""
tmux kill-session -t {sname} 2>/dev/null || true
tmux new-session -d -s {sname}
tmux send-keys -t {sname} 'cd ROBOTICS-lehome-challenge && source .venv/bin/activate && python scripts/mimicgen/leisaac_eef_action_process.py \
  --input_file {filepath} \
  --output_file {OUTPUT_DIR}/{out_stem}.hdf5 \
  --to_joint \
  --headless && tmux kill-session -t {sname}' Enter
"""
    ssh(cmd)
    print(f"  [STARTED] {sname}")
    return sname

queue = list(files)
launched = set()
completed = set()

while queue and len(launched - completed) < MAX_CONCURRENT_CONVERT:
    launched.add(launch_convert(queue.pop(0)))

while launched - completed:
    time.sleep(POLL_INTERVAL_CONVERT)
    active = active_sessions()
    for s in list(launched - completed - active):
        print(f"  [DONE] {s}")
        completed.add(s)
    while queue and len(launched - completed) < MAX_CONCURRENT_CONVERT:
        launched.add(launch_convert(queue.pop(0)))
    print(f"  active={len(launched - completed)}  done={len(completed)}  queued={len(queue)}")

print(f"\nAll {len(completed)} conversions completed.")

In [0]:
# %sh
# rsync -avz --progress \
#   -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
#   ubuntu@$BREV_IP:~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/4_generated_datasets/*.hdf5 \
#   /Volumes/workspace/default/hdf5datasets_lehome_many_clothes/

In [0]:
%sh
rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
  ubuntu@$BREV_IP:~/ROBOTICS-lehome-challenge/*.mp4 \
  /Volumes/workspace/default/hdf5datasets_lehome_many_clothes/

In [0]:
%sh
rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
  ubuntu@$BREV_IP:~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/5_final_generated_datasets/*.hdf5 \
  /Volumes/workspace/default/hdf5_test/